In [85]:
import pandas as pd
import numpy as np
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [86]:
df_raw = pd.read_csv('train.csv')

In [87]:
class CustomDataPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.financial_stress_median_ = None
        self.rare_jobs_ = None
        self.train_columns_ = None
        
    def fit(self, X, y=None):
        # Training appo mattum median matrum rare jobs-ai kandupudichu memory-la vechukkum
        self.financial_stress_median_ = X['Financial Stress'].median()
        
        job_counts = X['Profession'].value_counts()
        self.rare_jobs_ = job_counts[job_counts < 10].index.tolist()
        
        # Fit panrappe oru thadava transform panni, final columns list-ai save pannikkum (Streamlit-kaga)
        X_transformed = self._apply_transform(X, is_fit=True)
        self.train_columns_ = X_transformed.columns.tolist()
        return self
        
    def transform(self, X):
        return self._apply_transform(X, is_fit=False)
        
    def _apply_transform(self, X, is_fit=False):
        df = X.copy()
        
        # 1. Thevaiyillatha columns drop panrom
        cols_to_drop = ['id', 'Name', 'City', 'Degree']
        df = df.drop([c for c in cols_to_drop if c in df.columns], axis=1, errors='ignore')
        
        # 2. Binary Mappings
        gender_map = {'Female': 0, 'Male': 1}
        yes_no_map = {'No': 0, 'Yes': 1}
        work_stu_map = {'Student': 0, 'Working Professional': 1}
        
        df['Gender'] = df['Gender'].map(gender_map)
        df['Have you ever had suicidal thoughts ?'] = df['Have you ever had suicidal thoughts ?'].map(yes_no_map)
        df['Family History of Mental Illness'] = df['Family History of Mental Illness'].map(yes_no_map)
        df['Working Professional or Student'] = df['Working Professional or Student'].map(work_stu_map)
        
        # 3. Sleep Duration (VIP Whitelist & Mapping)
        valid_sleep = ['Less than 5 hours', '5-6 hours', '7-8 hours', 'More than 8 hours']
        df['Sleep Duration'] = df['Sleep Duration'].apply(lambda x: x if x in valid_sleep else '5-6 hours')
        sleep_map = {'Less than 5 hours': 0, '5-6 hours': 1, '7-8 hours': 2, 'More than 8 hours': 3}
        df['Sleep Duration'] = df['Sleep Duration'].map(sleep_map)
        
        # 4. Dietary Habits (VIP Whitelist & Mapping)
        valid_diet = ['Moderate', 'Unhealthy', 'Healthy']
        df['Dietary Habits'] = df['Dietary Habits'].apply(lambda x: x if x in valid_diet else 'Moderate')
        diet_map = {'Unhealthy': 0, 'Moderate': 1, 'Healthy': 2}
        df['Dietary Habits'] = df['Dietary Habits'].map(diet_map)
        
        # 5. Financial Stress -> Median Imputation
        df['Financial Stress'] = df['Financial Stress'].fillna(self.financial_stress_median_)
        
        # 6. Structural Zeros (0.0 filling)
        zero_cols = ['Academic Pressure', 'Study Satisfaction', 'CGPA', 'Work Pressure', 'Job Satisfaction']
        for col in zero_cols:
            df[col] = df[col].fillna(0.0)
            
        # 7. Profession Handling
        df['Profession'] = df['Profession'].fillna('Unemployed')
        df['Profession'] = df['Profession'].replace({'Finanancial Analyst': 'Financial Analyst'})
        df['Profession'] = df['Profession'].replace(self.rare_jobs_, 'Unemployed')
        
        # One-Hot Encoding
        df = pd.get_dummies(df, columns=['Profession'], drop_first=True, dtype=int)
        
        # CRITICAL STEP FOR STREAMLIT: User puthu data tharumbothu missing OHE columns-ai 0 aakkum
        if not is_fit:
            df = df.reindex(columns=self.train_columns_, fill_value=0)
            
        return df

In [88]:
X = df_raw.drop('Depression', axis=1)
y = df_raw['Depression'].values

In [89]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [90]:
master_pipeline = Pipeline([
    ('custom_cleaner', CustomDataPreprocessor()),
    ('scaler', StandardScaler())
])

In [91]:
print(X_train_raw.shape, X_test_raw.shape)
print(y_train.shape, y_test.shape)

(112560, 19) (28140, 19)
(112560,) (28140,)


In [92]:
X_train_processed = master_pipeline.fit_transform(X_train_raw)
X_test_processed = master_pipeline.transform(X_test_raw)

In [93]:
joblib.dump(master_pipeline, 'master_pipeline.pkl')
print(f"Pipeline saved successfully! Cleaned Data Shape: {X_train_processed.shape}")

Pipeline saved successfully! Cleaned Data Shape: (112560, 48)


In [94]:
X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

In [95]:
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=256, shuffle=True)

In [96]:
class DepressionMLP(nn.Module):
    def __init__(self,input_dim):
        super(DepressionMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()
            )
    def forward(self, x):
        return self.network(x)
    
input_size = X_train_processed.shape[1]
model = DepressionMLP(input_dim= input_size)
criterion = nn.BCELoss() # Binory Cross Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("\n-- Starting Neural Network Training --")
epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} | Average Loss: {total_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), 'depression_model.pth')
print("Model Saved.")

model.eval()
with torch.no_grad():
    test_predictions_prob = model(X_test_tensor)
    test_preds = (test_predictions_prob >= 0.5).float().numpy()
    
print("\n--- Final Model Performance ---")
print(f"Accuracy : {accuracy_score(y_test, test_preds):.4f}")
print(f"Precision: {precision_score(y_test, test_preds):.4f}")
print(f"Recall   : {recall_score(y_test, test_preds):.4f}")
print(f"F1-Score : {f1_score(y_test, test_preds):.4f}")
        


-- Starting Neural Network Training --
Epoch 1/10 | Average Loss: 0.2460
Epoch 2/10 | Average Loss: 0.1686
Epoch 3/10 | Average Loss: 0.1644
Epoch 4/10 | Average Loss: 0.1634
Epoch 5/10 | Average Loss: 0.1617
Epoch 6/10 | Average Loss: 0.1608
Epoch 7/10 | Average Loss: 0.1601
Epoch 8/10 | Average Loss: 0.1586
Epoch 9/10 | Average Loss: 0.1586
Epoch 10/10 | Average Loss: 0.1588
Model Saved.

--- Final Model Performance ---
Accuracy : 0.9386
Precision: 0.8478
Recall   : 0.8070
F1-Score : 0.8269


In [97]:
print(f"Accuracy : {accuracy_score(y_test, test_preds):.4f}")
print(f"Precision: {precision_score(y_test, test_preds):.4f}")
print(f"Recall   : {recall_score(y_test, test_preds):.4f}")
print(f"F1-Score : {f1_score(y_test, test_preds):.4f}")

Accuracy : 0.9386
Precision: 0.8478
Recall   : 0.8070
F1-Score : 0.8269
